# 02 — Filers

Reads `reference/filers.csv` and its provenance record, both produced by `scripts/02_select_filers.py`, and asks who the selection rule admits and what that costs.

This table is the join key for every other reference table and the population the pilot, the 250-claim annotation round that decides whether the full study is worth running samples from. A defect here does not surface as one wrong number, it changes what the study is about.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

REF = PROJECT_ROOT / "reference"
filers = pd.read_csv(REF / "filers.csv")
prov = json.loads((REF / "filers.provenance.json").read_text())
selection, screen = prov["selection"], prov["screen"]


def in_billions(frame: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with revenue rendered in billions, for reading."""
    out = frame.copy()
    out["revenue"] = (out["revenue"] / 1e9).round(1)
    return out.rename(columns={"revenue": "revenue_bn"})


print(f"{len(filers)} filers, built at commit {prov['commit']} on {prov['generated_at']}")
print(f"ranked by revenue at {selection['year']}, {selection['location_prefix']}* only")
for element, count in prov["candidates_per_element"].items():
    print(f"  {element:<58} {count:>5} candidates")
print(f"  {'after the location filter':<58} {prov['candidates_after_location_filter']:>5} candidates")

150 filers, built at commit 8b474ee on 2026-08-30T09:56:31Z
ranked by revenue at CY2023, US-* only
  us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax  3141 candidates
  us-gaap:Revenues                                            2677 candidates
  after the location filter                                   4214 candidates


## The rule

Filers are ranked by annual revenue from the SEC frames API and the largest 150 US filers are kept. A filer reporting under both revenue elements is counted once at the larger figure, because the two overlap during the ASC 606 transition and adding them would double-count.

Nothing is chosen by hand, which is the point: the set can be rebuilt from the rule, and the rule can be argued with.

In [2]:
print(f"revenue range: {filers['revenue'].min()/1e9:,.1f}B to {filers['revenue'].max()/1e9:,.0f}B")
print(f"median {filers['revenue'].median()/1e9:,.1f}B, mean {filers['revenue'].mean()/1e9:,.1f}B")
print(f"top ten hold {100*filers.head(10)['revenue'].sum()/filers['revenue'].sum():.0f}% of the set's revenue")
print(f"{filers['location'].nunique()} states or territories")
print()
for line in prov["limitations"]:
    print(f"- {line}")

revenue range: 25.5B to 648B
median 52.8B, mean 86.7B
top ten hold 30% of the set's revenue
29 states or territories

- Ranking at one recent year is a survivorship filter: filers large in 2012 that have since shrunk or delisted cannot appear.
- Revenue is a size proxy, not market capitalisation, and over-weights low-margin distribution: three drug wholesalers outrank Microsoft.
- Whether a financial firm appears is decided by tagging practice rather than size. Goldman Sachs, Morgan Stanley, Wells Fargo and Truist tag neither revenue element, so no cutoff admits them.
- The per-CIK dedupe cannot see one business filing under two CIKs, so several corporate families hold more than one slot.
- The definitive study set is this list intersected with the transcript corpus.


The set records its own limitations, and the rest of this notebook checks them against the table.

## The plausibility screen

The frames API serves whatever a filer tagged, scale errors included, so revenue over total assets is checked before the ranking is taken. A candidate that fails is excluded rather than flagged: an implausible revenue figure does not make a filer suspect, it makes its rank fabricated.

In [3]:
print(f"threshold: revenue must not exceed {screen['max_revenue_to_assets']:.0f}x total assets")
print(f"assets from {screen['assets_element']} at {screen['assets_period']}, pool of {screen['pool_size']}")
print()
print(f"kept, highest ratio  {filers['revenue_to_assets'].max():>10,.1f}x   {filers.loc[filers['revenue_to_assets'].idxmax(), 'name']}")
print(f"kept, median ratio   {filers['revenue_to_assets'].median():>10,.2f}x")
print()
for item in screen["excluded"]:
    print(f"excluded             {item['revenue_to_assets']:>10,.0f}x   {item['name']}")
    print(f"    revenue {item['revenue']:>18,.0f}")
    print(f"    assets  {item['assets']:>18,.0f}")
    print(f"    would have ranked {item['would_have_ranked']}")

threshold: revenue must not exceed 25x total assets
assets from us-gaap:Assets at CY2023Q4I, pool of 200

kept, highest ratio         6.5x   WORLD KINECT CORPORATION
kept, median ratio         0.78x

excluded                  1,137x   Tigo Energy, Inc.
    revenue    145,233,000,000
    assets         127,777,000
    would have ranked 25


Tigo Energy tagged CY2023 revenue as 145,233,000,000 against total assets of 127,777,000, in its own 10-K, filed in March 2024 and tagged the same way again in March 2025. Revenue of 1,137 times total assets is not a business, it is a scale error: the true figure is 145.233 million and the tag is off by a factor of a thousand. It would have ranked 25th, between Phillips 66 and Meta.

The threshold sits at 25 rather than in the middle of the gap it has to cross. Across the top 300 candidates the median ratio is 0.70 and the largest genuine reading is 6.5, for World Kinect, a fuel distributor turning enormous throughput on a small balance sheet. Nothing lies between that and Tigo. A thousand-fold error on an asset-heavy filer, a bank or a utility whose true ratio is near 0.05, would land around 50, so a higher cut would let those through while still catching this one.

Two selected filers have no assets figure to screen against, and are kept. Absence of a balance sheet to check is not evidence of an error, and their CIKs are in the provenance so the gap is visible rather than assumed away.

In [4]:
unscreened = filers[filers["assets"].isna()]
print(f"{len(unscreened)} of {len(filers)} kept without a screen:\n")
print(in_billions(unscreened)[["rank", "cik", "name", "revenue_bn"]].to_string(index=False))

2 of 150 kept without a screen:

 rank     cik                           name  revenue_bn
   77 1859392            Galaxy Digital Inc.        51.6
  135 2041610 Paramount Skydance Corporation        29.7


## Revenue ranks throughput, not size

In [5]:
in_billions(filers.head(15))[["rank", "name", "location", "revenue_bn", "revenue_to_assets"]].set_index("rank")

,name,location,revenue_bn,revenue_to_assets
rank,,,,
1,Walmart Inc.,US-AR,648.1,2.568
2,"AMAZON.COM, INC.",US-WA,574.8,1.089
3,Apple Inc.,US-CA,383.3,1.084
4,UnitedHealth Group Incorporated,US-MN,371.6,1.358
5,BERKSHIRE HATHAWAY INC,US-NE,364.5,0.341
6,CVS HEALTH CORPORATION,US-RI,357.8,1.433
7,Exxon Mobil Corporation,US-TX,344.6,0.916
8,McKESSON CORPORATION,US-TX,309.0,4.645
9,Alphabet Inc.,US-CA,307.4,0.764


Three drug distributors sit in the top thirteen: McKesson at 8, Cencora at 10 and Cardinal Health at 13, all of them above Microsoft at 12. The ratio column says why. Those three turn 4.0 to 4.6 times their balance sheet in a year while Microsoft turns 0.45, so ranking by revenue puts wholesalers where a market-value ranking would put software.

If claim behaviour varies by sector, this sample leans toward distribution, retail and managed care. That is a covariate to record at analysis rather than something to correct here, because any correction would be a hand adjustment to a rule whose value is that it is mechanical.

## Which financial firms the rule keeps

In [6]:
# Checked by CIK. SEC entity names are not stable: U.S. Bancorp files as
# "US BANCORP \DE\", so a name join reports it missing when it is in the set.
INSTITUTIONS = {
    19617: "JPMorgan Chase", 70858: "Bank of America", 831001: "Citigroup",
    1099219: "MetLife", 1137774: "Prudential Financial", 4962: "American Express",
    927628: "Capital One", 5272: "AIG", 36104: "U.S. Bancorp",
    886982: "Goldman Sachs", 895421: "Morgan Stanley", 72971: "Wells Fargo",
    92230: "Truist", 316709: "Charles Schwab", 713676: "PNC", 1364742: "BlackRock",
}

ranks = filers.set_index("cik")["rank"]
membership = pd.DataFrame(
    [{"institution": name, "cik": cik, "rank": ranks.get(cik)} for cik, name in INSTITUTIONS.items()]
)
membership["rank"] = membership["rank"].astype("Int64")

inside = membership["rank"].notna().sum()
print(f"{inside} of {len(membership)} in the study, {len(membership) - inside} not")
print()
membership.sort_values("rank", na_position="last").reset_index(drop=True)

9 of 16 in the study, 7 not



,institution,cik,rank
0,JPMorgan Chase,19617,19
1,Bank of America,70858,32
2,Citigroup,831001,46
3,MetLife,1099219,52
4,Prudential Financial,1137774,71
5,American Express,4962,99
6,Capital One,927628,100
7,U.S. Bancorp,36104,140
8,AIG,5272,141
9,Goldman Sachs,886982,<NA>


Nine of the sixteen are in, JPMorgan highest at 19. The seven that are not split into two cases, and only three of them are the rule working.

Charles Schwab, PNC and BlackRock report revenue below the cutoff and are correctly excluded. Goldman Sachs, Morgan Stanley, Wells Fargo and Truist appear in neither revenue frame for CY2023, so no cutoff would have admitted them: banks that report interest and non-interest income rather than a revenue line are invisible to this rule at any size.

Financial-sector coverage is therefore set by tagging practice, not by size, which is what the second limitation in the provenance now says.

## Related entities holding separate slots

In [7]:
exact = filers[filers.duplicated("revenue", keep=False)]
print(f"pairs sharing an identical revenue figure: {len(exact) // 2}")
print()
print(in_billions(exact.sort_values(["revenue", "rank"], ascending=[False, True]))
      [["rank", "cik", "name", "revenue_bn"]].to_string(index=False))

token = filers["name"].str.upper().str.replace(r"[^A-Z ]", "", regex=True).str.split().str[0]
shared = filers[token.isin(token[token.duplicated()])]
print()
print(f"filers sharing a first name token: {len(shared)}")
print()
print(in_billions(shared.sort_values("name"))[["rank", "cik", "name", "revenue_bn"]].to_string(index=False))

pairs sharing an identical revenue figure: 3

 rank     cik                            name  revenue_bn
   67 1091667    Charter Communications, Inc.        54.6
   68 1271833               CCO Holdings, LLC        54.6
   82 1070423 PLAINS ALL AMERICAN PIPELINE LP        47.3
   83 1581990           PLAINS GP HOLDINGS LP        47.3
  134  813828                Paramount Global        29.7
  135 2041610  Paramount Skydance Corporation        29.7

filers sharing a first name token: 19

 rank     cik                               name  revenue_bn
   76    6201       American Airlines Group Inc.        52.8
   99    4962                American Express Co        37.2
  141    5272 American International Group, Inc.        27.9
  149 1081316  BERKSHIRE HATHAWAY ENERGY COMPANY        25.6
    5 1067983             BERKSHIRE HATHAWAY INC       364.5
   90   40533       GENERAL DYNAMICS CORPORATION        42.3
  108   40545           GENERAL ELECTRIC COMPANY        35.3
   17 1467858       

Six corporate families hold twelve of the 150 slots: Charter with CCO Holdings, Plains All American with Plains GP, PBF Energy with PBF Holding, Paramount Global with Paramount Skydance, MetLife with Metropolitan Life, and Berkshire Hathaway with Berkshire Hathaway Energy.

The dedupe inside `select()` is per CIK, which is the right rule for one filer reporting under two elements and no rule at all for one business filing under two CIKs.

No mechanical screen finds all six. Exact revenue equality finds three and raises no false positives. A shared first name token adds PBF and Berkshire, along with unrelated matches: American Airlines, American Express and AIG share nothing but a word. MetLife and its own subsidiary are caught by neither, and were found by reading the table.

Most of these subsidiaries file because they carry public debt and hold no earnings call. The study set is this list intersected with the earnings-call transcript corpus, since a filer with no transcript contributes no claims, so that step removes them without a new rule. Paramount is the case that needs a decision, because a successor entity can inherit the call series and the same claim would then be adjudicated twice.

## Where the cutoff sits

In [8]:
cutoff = filers["revenue"].min()
print(f"rank {len(filers)} sits at {cutoff/1e9:,.1f}B")
print()
for margin in (0.05, 0.10, 0.20):
    within = (filers["revenue"] <= cutoff * (1 + margin)).sum()
    print(f"  within {margin:>4.0%} of the cutoff: {within:>3} filers")

rank 150 sits at 25.5B

  within   5% of the cutoff:   7 filers
  within  10% of the cutoff:  10 filers
  within  20% of the cutoff:  20 filers


Ten filers sit within a tenth of the boundary, twenty within a fifth, so `FILER_COUNT` is a dial rather than a natural break. Moving it from 150 to 130 or 170 changes roughly a fifth of the names, and changing it rebuilds every table downstream that joins on `cik`.

## Which element supplied the figure

In [9]:
filers["source_element"].value_counts().rename("filers").to_frame()

,filers
source_element,
us-gaap:Revenues,81
us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax,69


The split is close to even, and it is the same ASC 606 boundary [01_metric_classes.ipynb](01_metric_classes.ipynb) found in the metric table. Neither element spans 2012 to 2024 on its own: the contract-with-customer element does not exist at the start of the window and `Revenues` is being retired across it. Ranking on either one alone would have produced a different set of 150 for a reason that has nothing to do with size.

## What this implies for the pilot

**The screen removed one filer and no others were close.** Tigo Energy would have ranked 25th on a thousand-fold tagging error. The highest ratio among the 150 kept is 6.5, so the threshold is nowhere near a real business, and the exclusion is named in the provenance rather than being a silent gap in the ranking.

**Twelve slots go to six corporate families.** The per-CIK dedupe cannot see this. Intersecting with the transcript corpus removes the debt-issuing subsidiaries for free, since they hold no earnings call; Paramount Global against Paramount Skydance needs a decision before sampling, or one claim gets adjudicated twice.

**Four large banks are unreachable by this rule at any cutoff.** Goldman Sachs, Morgan Stanley, Wells Fargo and Truist tag neither revenue element. Anything this study says about financial-sector claims is conditioned on tagging practice, which is worth stating in the paper rather than discovering in review.

**Revenue ranks throughput.** Three drug distributors outrank Microsoft, and the ratio column shows the mechanism. Sector composition is a covariate to carry into the analysis, not a defect to correct by hand.

**CIK is the only safe join key.** `US BANCORP \DE\` and `SCHWAB CHARLES CORP` are what the SEC calls those filers. Any join on name silently loses rows.